# 🫁 ResNet-50 CXR — Clasificación Multilabel (v3 — Optimizado para CPU)
## TFM: Sistema de Apoyo a la Decisión Clínica Multimodal · Módulo de Imagen
### Universidad de Salamanca · Máster en Análisis Avanzado de Datos Multivariantes y Big Data

---

## 📋 Descripción general

Este notebook implementa el **modelo de imagen** (radiografías de tórax, CXR) del sistema
multimodal sobre el dataset **Symile-MIMIC**, en su versión **v3 optimizada para hardware
sin GPU dedicada** (probado sobre Intel i5-1135G7, 4 núcleos, sin CUDA).

### Hardware objetivo y su impacto en las decisiones de diseño
- **CPU**: Intel Core i5-1135G7 @ 2.4GHz, 4 núcleos físicos / 8 hilos, sin GPU NVIDIA.
- Un ResNet-50 con imágenes de 320×320 tardaba **~47s/batch** → inviable para Nested CV.
- Este notebook reduce el coste en **~15-20× adicionales** sobre la v2, manteniendo
  la validez metodológica del Nested CV (sigue habiendo separación train/val/test y
  selección de hiperparámetros independiente de la evaluación).

### Fuente de las imágenes
Las imágenes CXR se leen desde `cxr_train.npy / cxr_val.npy / cxr_test.npy`
(arrays float32 pre-normalizados, shape `(N, 3, 320, 320)`), **no** desde JPGs en disco.
La columna `cxr_path` del CSV es solo un identificador; el join real es por `hadm_id`.

### Lista completa de fixes acumulados (v1 → v3)

| Fix | Origen | Descripción |
|-----|--------|-------------|
| **[FIX 1]** | v2 | Imágenes leídas desde `.npy` en memoria, no JPGs en disco |
| **[FIX 2]** | v2 | Join CSV↔npy por `hadm_id` → columna `_npy_idx` |
| **[FIX 2b]**| v2 | Test CSV=464 filas vs npy=4640 (1/10 del original) — resuelto por el join |
| **[FIX 3]** | v1 | `ReduceLROnPlateau`: eliminado `verbose=False` (deprecado PyTorch ≥2.2) |
| **[FIX 4]** | v1 | `build_optimizer_and_scheduler` con `num_epochs_override` (evita overflow OneCycleLR) |
| **[FIX 5]** | v1 | Guardia NaN en `macro_AUC` cuando no hay positivos suficientes |
| **[FIX 6]** | v1 | `gc.collect()` + `cuda.empty_cache()` combinados |
| **[FIX 7]** | v1 | Recalculo de `auc_macros`/`f1_macros` al inicio de cada celda (scope Jupyter) |
| **[FIX 8]** | v2→v3 | Progreso explícito con `print(..., flush=True)` + ETA (tqdm no fiable en loops anidados bloqueantes) |
| **[FIX 9]** | v2 | Imágenes ya normalizadas en el npy — sin `A.Normalize`, sin conversión a uint8 |
| **[FIX 10]**| v2 | Colapso 3→1 canal en `forward()` (backbone CheXpert espera grayscale) |
| **[FIX 11]**| v2 | Detección dinámica de la dimensión de salida del backbone XRV (2048, sin GAP externo) |
| **[FIX 12]**| v2 | Extracción del backbone puro de `xrv_model.model` quitando `avgpool+fc` |
| **[FIX 13]**| **v3 (nuevo)** | `IMG_SIZE` 320→**160**: 4× menos píxeles por imagen |
| **[FIX 14]**| **v3 (nuevo)** | Subsampling al **15%** del train en el loop interno del Nested CV |
| **[FIX 15]**| **v3 (nuevo)** | `batch_size` fijado a **8** (mejor uso de caché L2/L3 en CPU sin GPU) |
| **[FIX 16]**| **v3 (nuevo)** | `unfreeze_layers` fijado a `"none"` — backbone permanece congelado siempre (solo se entrena la cabeza) |
| **[FIX 17]**| **v3 (nuevo)** | `num_epochs` reducido a 3 con paciencia 1 (early stopping agresivo) |
| **[FIX 18]**| **v3 (nuevo)** | `N_RANDOM_CONFIGS` 6→**3**, `K_OUTER` 3→**2** |
| **[FIX 19]**| **v3 (nuevo)** | `torch.set_num_threads()` fijado explícitamente a los núcleos físicos disponibles |
| **[FIX 20]**| **v3 (nuevo)** | Cache de embeddings: el backbone congelado se ejecuta **una sola vez** por imagen y se cachea, evitando recomputar el forward del backbone en cada época |

### ⚠️ Nota metodológica importante sobre [FIX 14] y [FIX 16]
- El subsampling al 15% en el loop interno **solo afecta a la selección de
  hiperparámetros**; el reentrenamiento final de cada fold externo usa el 100% de
  `train_outer`. Esto es una práctica aceptada para hacer viable el tuning sin
  comprometer la estimación final de rendimiento.
- Congelar el backbone permanentemente ([FIX 16]) convierte el ResNet-50 en un
  **extractor de features fijo** (transfer learning "feature extraction" puro,
  en vez de fine-tuning). Es una simplificación deliberada: se sacrifica algo de
  rendimiento potencial a cambio de viabilidad computacional. Esto se debe
  documentar explícitamente en la memoria del TFM como limitación del entorno.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 1: INSTALACIÓN DE DEPENDENCIAS
# ══════════════════════════════════════════════════════════════════════════════

import subprocess, sys

packages = [
    "torchxrayvision",
    "albumentations",
    "scikit-learn",
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "tqdm",
    "Pillow",
]

for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=True)

print("✅ Dependencias instaladas.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 2: IMPORTACIONES, SEMILLAS Y CONFIGURACIÓN DE HILOS CPU
# [FIX 19] torch.set_num_threads fijado a los núcleos físicos reales.
#          Por defecto PyTorch puede sobre-suscribir hilos (hyperthreading)
#          y degradar el rendimiento en CPUs de portátil con throttling térmico.
# ══════════════════════════════════════════════════════════════════════════════

import os, gc, warnings, random, json, copy, time
from pathlib import Path
from itertools import product as itertools_product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.models as tv_models

try:
    import torchxrayvision as xrv
    XRV_AVAILABLE = True
except ImportError:
    XRV_AVAILABLE = False
    print("⚠ torchxrayvision no disponible — se usará ResNet-50 ImageNet como fallback.")

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

warnings.filterwarnings("ignore")

# ── Reproducibilidad ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# [FIX 19] Fijar hilos CPU a los núcleos FÍSICOS (i5-1135G7 = 4 físicos, 8 lógicos).
# Usar más hilos que núcleos físicos para operaciones BLAS suele ser contraproducente
# en portátiles por el hyperthreading y el throttling térmico bajo carga sostenida.
N_PHYSICAL_CORES = 4
torch.set_num_threads(N_PHYSICAL_CORES)
os.environ["OMP_NUM_THREADS"]      = str(N_PHYSICAL_CORES)
os.environ["MKL_NUM_THREADS"]      = str(N_PHYSICAL_CORES)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Dispositivo : {DEVICE}")
print(f"   PyTorch     : {torch.__version__}")
print(f"   Hilos CPU   : {torch.get_num_threads()}  [FIX 19]")
print(f"   XRV         : {'disponible' if XRV_AVAILABLE else 'NO disponible (fallback ImageNet)'}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 3: CONFIGURACIÓN DE RUTAS Y CONSTANTES GLOBALES
# [FIX 1] Imágenes desde .npy, no JPGs en disco.
# [FIX 13] IMG_SIZE reducido 320→160 (4× menos píxeles, ~4× menos cómputo conv).
# ══════════════════════════════════════════════════════════════════════════════

BASE_DATA_DIR = Path(
    r"C:\TFM\1.OPCIÓN - SYMILE MIMIC\SYMILE-MIMIC-A-MULTIMODAL-CLINICAL-DATASET-OF-CHEST-X-RAYS-ELECTROCARDIOGRAMS-AND-BLOOD-LABS-FROM-MIMIC-IV-1.0.0"
)

CSV_DIR   = BASE_DATA_DIR / "data_csv" / "clean"
TRAIN_CSV = CSV_DIR / "train_clean.csv"
VAL_CSV   = CSV_DIR / "val_clean.csv"
TEST_CSV  = CSV_DIR / "test_clean.csv"

NPY_DIR       = BASE_DATA_DIR / "data_npy"
CXR_TRAIN_NPY = NPY_DIR / "train" / "cxr_train.npy"
CXR_VAL_NPY   = NPY_DIR / "val"   / "cxr_val.npy"
CXR_TEST_NPY  = NPY_DIR / "test"  / "cxr_test.npy"

HADM_TRAIN_NPY = NPY_DIR / "train" / "hadm_id_train.npy"
HADM_VAL_NPY   = NPY_DIR / "val"   / "hadm_id_val.npy"
HADM_TEST_NPY  = NPY_DIR / "test"  / "hadm_id_test.npy"

OUTPUT_DIR = Path("outputs_resnet50_cxr_v3")
OUTPUT_DIR.mkdir(exist_ok=True)

LABELS   = ["Atelectasis", "Cardiomegaly", "Edema", "Lung Opacity", "No Finding", "Pleural Effusion"]
N_LABELS = len(LABELS)

POS_WEIGHTS = {
    "Atelectasis"     : 0.04,
    "Cardiomegaly"    : 0.30,
    "Edema"           : 0.90,
    "Lung Opacity"    : 0.10,
    "No Finding"      : 1.00,
    "Pleural Effusion": 0.50,
}

# [FIX 13] 320 → 160. El npy original es 320×320; albumentations lo redimensiona
# a 160×160 en el pipeline. Esto reduce el coste de las convoluciones en ~4×
# (área proporcional al cuadrado del lado).
IMG_SIZE = 160

GENDER_MAP    = {0: 0, 1: 1}
RACE_MAP      = {"UNKNOWN": 0, "WHITE": 1, "BLACK": 2, "ASIAN": 3, "HISPANIC_LATINO": 4}
ADMISSION_MAP = {"SCHEDULED": 0, "EMERGENCY": 1, "OBSERVATION": 2, "URGENT": 3}
CXR_VIEW_MAP  = {"AP": 0, "PA": 1}
META_DIM      = 13
META_EMBED    = 64
IMG_PROJ      = 512

print("✅ Constantes configuradas.")
print(f"   [FIX 13] IMG_SIZE = {IMG_SIZE} (reducido desde 320)")
print(f"   CXR train npy : {CXR_TRAIN_NPY}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 4: CARGA DE DATOS Y ALINEACIÓN CSV ↔ NPY
# [FIX 1][FIX 2][FIX 2b] Join por hadm_id, mmap_mode="r" para no cargar
# ~11GB en RAM de golpe (importante en equipos con memoria limitada).
# ══════════════════════════════════════════════════════════════════════════════

def load_split(csv_path, cxr_npy_path, hadm_npy_path, split_name="train"):
    """
    Carga un split completo alineando CSV y npy por hadm_id.
    mmap_mode="r": el array se mapea desde disco, no se carga entero en RAM.
    Esto es crítico en equipos con poca memoria (portátiles de gama media).
    """
    print(f"\n── Cargando split '{split_name}' ────────────────────────────────────")

    df = pd.read_csv(csv_path, sep=";")
    print(f"   CSV filas          : {len(df):,}")

    hadm_npy = np.load(hadm_npy_path, allow_pickle=True)
    print(f"   npy hadm_id count  : {len(hadm_npy):,}")

    hadm_to_npy_idx = {int(h): i for i, h in enumerate(hadm_npy)}

    df["_npy_idx"] = df["hadm_id"].apply(lambda h: hadm_to_npy_idx.get(int(h), -1))
    n_before = len(df)
    df = df[df["_npy_idx"] >= 0].reset_index(drop=True)
    print(f"   Filas con imagen   : {len(df):,}  (descartadas: {n_before - len(df)})")

    print(f"   Mapeando {cxr_npy_path.name} (mmap, sin carga completa en RAM) ...",
          end=" ", flush=True)
    cxr_npy = np.load(cxr_npy_path, mmap_mode="r")
    print(f"shape={cxr_npy.shape}  dtype={cxr_npy.dtype}")

    return df, cxr_npy, hadm_to_npy_idx


df_train, cxr_train_npy, hadm_to_npy_train = load_split(TRAIN_CSV, CXR_TRAIN_NPY, HADM_TRAIN_NPY, "train")
df_val,   cxr_val_npy,   hadm_to_npy_val   = load_split(VAL_CSV,   CXR_VAL_NPY,   HADM_VAL_NPY,   "val")
df_test,  cxr_test_npy,  hadm_to_npy_test  = load_split(TEST_CSV,  CXR_TEST_NPY,  HADM_TEST_NPY,  "test")

print(f"\n✅ Datos cargados:")
print(f"   Train : {len(df_train):,} muestras  |  CXR npy shape: {cxr_train_npy.shape}")
print(f"   Val   : {len(df_val):,}  muestras  |  CXR npy shape: {cxr_val_npy.shape}")
print(f"   Test  : {len(df_test):,}   muestras  |  CXR npy shape: {cxr_test_npy.shape}")
print(f"   [FIX 2b] Test CSV={len(df_test)} filas alineadas con npy={cxr_test_npy.shape[0]} → join correcto.")

sample = np.array(cxr_train_npy[0])  # forzar lectura desde mmap para inspección
print(f"\n   Muestra npy[0]: shape={sample.shape}  min={sample.min():.1f}  "
      f"max={sample.max():.1f}  mean={sample.mean():.2f}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 5: GRID DE HIPERPARÁMETROS — VERSIÓN MÍNIMA PARA CPU GAMA MEDIA
# ══════════════════════════════════════════════════════════════════════════════
# [FIX 16] unfreeze_layers fijado a "none": el backbone NUNCA se descongela.
#          Esto convierte el entrenamiento en "feature extraction" puro:
#          solo se actualizan img_proj + head + label_corr + meta_branch.
#          Reduce drásticamente el coste del backward pass (no hay gradiente
#          a través de las 50 capas del ResNet).
# [FIX 17] num_epochs=3 con paciencia=1 (ver celda 11).
# [FIX 18] N_RANDOM_CONFIGS 6→3.
# ══════════════════════════════════════════════════════════════════════════════

HYPERPARAM_GRID = {
    "lr_backbone"          : [1e-5],          # Irrelevante si backbone congelado, pero se mantiene por compatibilidad
    "lr_head"               : [1e-4, 3e-4],
    "scheduler"              : ["cosine"],
    "num_epochs"             : [3],            # [FIX 17] antes 6, ahora 3
    "unfreeze_epoch"         : [999],          # [FIX 16] nunca se alcanza → backbone siempre congelado
    "unfreeze_layers"        : ["none"],       # [FIX 16]
    "uncertainty_policy"     : ["zeros", "ones"],
    "dropout_rate"           : [0.3, 0.5],
    "weight_decay"           : [1e-4],
    "batch_size"             : [8],            # [FIX 15] reducido de 32 a 8
    "use_label_correlation"  : [True, False],
    "use_meta_branch"        : [True],
    "threshold_search"       : [True],
    "augmentation_level"     : ["moderate"],
}

N_RANDOM_CONFIGS = 3   # [FIX 18] antes 6

all_keys   = list(HYPERPARAM_GRID.keys())
all_values = list(HYPERPARAM_GRID.values())
all_combos = list(itertools_product(*all_values))
np.random.shuffle(all_combos)
SAMPLED_CONFIGS = [dict(zip(all_keys, c)) for c in all_combos[:N_RANDOM_CONFIGS]]

print(f"✅ Grid MÍNIMO definido para viabilidad en CPU sin GPU.")
print(f"   Combinaciones posibles : {len(all_combos)}")
print(f"   Configs muestreadas    : {N_RANDOM_CONFIGS}")
print(f"   [FIX 16] Backbone SIEMPRE congelado (feature extraction puro)")
print(f"   [FIX 15] batch_size=8  |  [FIX 17] num_epochs=3")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 6: PIPELINES DE AUGMENTACIÓN
# [FIX 9] Sin A.Normalize: las imágenes del npy ya vienen normalizadas.
# [FIX 13] A.Resize usa IMG_SIZE=160 en lugar de 320.
# ══════════════════════════════════════════════════════════════════════════════

def get_augmentation_pipeline(level: str, img_size: int = IMG_SIZE):
    """
    [FIX 9] No incluye A.Normalize — las imágenes del npy ya están normalizadas
    (rango aproximado [-2.1, 2.6], convenio ImageNet aplicado en el preprocesado
    de Symile-MIMIC).
    [FIX 13] img_size por defecto = 160 (antes 320) para reducir coste convolucional.
    """
    to_tensor = ToTensorV2()

    if level == "test":
        return A.Compose([A.Resize(img_size, img_size), to_tensor])

    elif level == "moderate":
        return A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=10, p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
            to_tensor,
        ])

    elif level == "aggressive":
        # Conservado por compatibilidad con el grid completo, aunque v3 no lo usa
        # por defecto (HYPERPARAM_GRID solo incluye "moderate" para reducir coste).
        return A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=15, p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.5),
            A.GaussNoise(var_limit=(0.001, 0.005), p=0.3),
            to_tensor,
        ])

    else:
        raise ValueError(f"Nivel desconocido: '{level}'.")


print("✅ Pipelines de augmentación definidos (sin Normalize, img_size=160).")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 7: DATASET PYTORCH — CXR MULTILABEL DESDE NPY
# [FIX 1] Lectura desde npy en memoria/mmap.
# [FIX 9] npy_to_hwc_float: transpone sin re-normalizar.
# ══════════════════════════════════════════════════════════════════════════════

def npy_to_hwc_float(arr: np.ndarray) -> np.ndarray:
    """
    Convierte (C,H,W) o (H,W) a (H,W,C) float32, SIN normalizar
    (las imágenes ya vienen normalizadas del npy de Symile-MIMIC).
    """
    arr = np.array(arr, dtype=np.float32)  # materializa desde mmap si aplica

    if arr.ndim == 3 and arr.shape[0] in (1, 3):
        arr = arr.transpose(1, 2, 0)
    elif arr.ndim == 2:
        arr = np.stack([arr, arr, arr], axis=-1)

    if arr.shape[2] == 1:
        arr = np.repeat(arr, 3, axis=2)

    return arr


class CXRMultilabelDataset(Dataset):
    """
    [FIX 1] Lee imágenes desde npy usando _npy_idx (columna calculada en load_split).
    [FIX 9] No re-normaliza.
    [FIX 20] Soporta un caché de embeddings opcional (ver CachedEmbeddingDataset
             en celda 7b) para evitar recomputar el backbone congelado.
    """

    def __init__(self, dataframe, cxr_npy, augmentation_pipeline,
                 uncertainty_policy="zeros", labels=LABELS):
        self.df                 = dataframe.reset_index(drop=True)
        self.cxr_npy            = cxr_npy
        self.transform          = augmentation_pipeline
        self.uncertainty_policy = uncertainty_policy
        self.labels             = labels

        age_col      = self.df["age"].astype(float)
        self.age_min = age_col.min()
        self.age_max = age_col.max()

    def __len__(self):
        return len(self.df)

    def _encode_labels_and_mask(self, row):
        label_vec = np.zeros(len(self.labels), dtype=np.float32)
        mask_vec  = np.zeros(len(self.labels), dtype=np.float32)
        for i, lbl in enumerate(self.labels):
            val = row[lbl]
            if pd.isna(val):
                label_vec[i], mask_vec[i] = 0.0, 0.0
            elif val == -1:
                mask_vec[i]  = 1.0
                label_vec[i] = 1.0 if self.uncertainty_policy == "ones" else 0.0
            else:
                label_vec[i] = float(val)
                mask_vec[i]  = 1.0
        return label_vec, mask_vec

    def _encode_metadata(self, row):
        age_norm = (float(row["age"]) - self.age_min) / (self.age_max - self.age_min + 1e-8)
        gender   = float(GENDER_MAP.get(row["gender"], 0))
        race_vec = np.zeros(len(RACE_MAP),      dtype=np.float32)
        adm_vec  = np.zeros(len(ADMISSION_MAP), dtype=np.float32)
        view_vec = np.zeros(len(CXR_VIEW_MAP),  dtype=np.float32)
        race_vec[RACE_MAP.get(str(row["race"]), 0)]               = 1.0
        adm_vec[ADMISSION_MAP.get(str(row["admission_type"]), 1)] = 1.0
        view_vec[CXR_VIEW_MAP.get(str(row["cxr_view"]), 0)]      = 1.0
        return np.concatenate([[age_norm, gender], race_vec, adm_vec, view_vec])

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        npy_idx = int(row["_npy_idx"])
        img_arr = self.cxr_npy[npy_idx]
        image   = npy_to_hwc_float(img_arr)

        image_tensor = self.transform(image=image)["image"].float()

        labels, mask = self._encode_labels_and_mask(row)
        metadata     = self._encode_metadata(row)

        return {
            "image"   : image_tensor,
            "labels"  : torch.tensor(labels,   dtype=torch.float32),
            "mask"    : torch.tensor(mask,      dtype=torch.float32),
            "metadata": torch.tensor(metadata,  dtype=torch.float32),
            "hadm_id" : int(row["hadm_id"]),
        }


print("✅ CXRMultilabelDataset definido (lectura npy + sin re-normalización).")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 7b: CACHÉ DE EMBEDDINGS DEL BACKBONE CONGELADO
# [FIX 20] Como el backbone NUNCA se descongela (FIX 16), su salida para una
#          imagen dada es SIEMPRE la misma (sin augmentación geométrica el
#          embedding es idéntico en cada época). En vez de recomputar el
#          forward de las 50 capas del ResNet en cada época, lo calculamos
#          UNA SOLA VEZ por split y lo guardamos en memoria.
#
#          ⚠ Esto es compatible con augmentación SOLO si la augmentación se
#          considera parte del backbone (lo cual no es el caso aquí: usamos
#          augmentación geométrica/intensidad que SÍ cambia la imagen de
#          entrada). Por tanto, el caché se calcula sobre la imagen SIN
#          augmentación aleatoria (equivalente a 'test' pipeline) — esto es
#          una simplificación adicional deliberada para v3: se sacrifica la
#          augmentación durante el entrenamiento de la cabeza a cambio de
#          una reducción de tiempo de ~10-15× (el backbone es la parte más
#          cara computacionalmente).
#
#          Esta es la optimización de mayor impacto del notebook: convierte
#          el problema de "entrenar un ResNet-50" en "entrenar una MLP
#          pequeña sobre features ya extraídas", con coste casi nulo en CPU.
# ══════════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def compute_embeddings_cache(df, cxr_npy, backbone, batch_size=16, desc="Cache"):
    """
    Calcula el embedding [N, 2048] de TODAS las imágenes de un DataFrame usando
    el backbone (congelado) UNA SOLA VEZ, sin augmentación aleatoria.

    Returns:
        embeddings: np.ndarray [N, 2048] float32
    """
    backbone.eval()
    aug_fixed = get_augmentation_pipeline("test")  # sin aleatoriedad
    ds_fixed  = CXRMultilabelDataset(df, cxr_npy, aug_fixed, uncertainty_policy="zeros")
    loader    = DataLoader(ds_fixed, batch_size=batch_size, shuffle=False, num_workers=0)

    all_embeds = []
    pbar = tqdm(loader, desc=desc, leave=False, unit="batch")
    for batch in pbar:
        images = batch["image"]
        if images.shape[1] == 3:
            images = images.mean(dim=1, keepdim=True)  # [FIX 10] 3→1 canal

        out = backbone(images)
        if out.ndim == 4:
            out = F.adaptive_avg_pool2d(out, 1).flatten(1)
        elif out.ndim == 3:
            out = out.mean(dim=-1)
        all_embeds.append(out.numpy())

    return np.concatenate(all_embeds, axis=0)


class CachedEmbeddingDataset(Dataset):
    """
    [FIX 20] Dataset que devuelve embeddings PRE-CALCULADOS en vez de imágenes.
    Permite entrenar la cabeza clasificadora sin tocar el backbone en cada paso.
    """
    def __init__(self, dataframe, embeddings, uncertainty_policy="zeros", labels=LABELS):
        self.df          = dataframe.reset_index(drop=True)
        self.embeddings  = embeddings  # [N, 2048] alineado por posición con self.df
        self.uncertainty_policy = uncertainty_policy
        self.labels      = labels

        age_col      = self.df["age"].astype(float)
        self.age_min = age_col.min()
        self.age_max = age_col.max()

    def __len__(self):
        return len(self.df)

    def _encode_labels_and_mask(self, row):
        label_vec = np.zeros(len(self.labels), dtype=np.float32)
        mask_vec  = np.zeros(len(self.labels), dtype=np.float32)
        for i, lbl in enumerate(self.labels):
            val = row[lbl]
            if pd.isna(val):
                label_vec[i], mask_vec[i] = 0.0, 0.0
            elif val == -1:
                mask_vec[i]  = 1.0
                label_vec[i] = 1.0 if self.uncertainty_policy == "ones" else 0.0
            else:
                label_vec[i] = float(val)
                mask_vec[i]  = 1.0
        return label_vec, mask_vec

    def _encode_metadata(self, row):
        age_norm = (float(row["age"]) - self.age_min) / (self.age_max - self.age_min + 1e-8)
        gender   = float(GENDER_MAP.get(row["gender"], 0))
        race_vec = np.zeros(len(RACE_MAP),      dtype=np.float32)
        adm_vec  = np.zeros(len(ADMISSION_MAP), dtype=np.float32)
        view_vec = np.zeros(len(CXR_VIEW_MAP),  dtype=np.float32)
        race_vec[RACE_MAP.get(str(row["race"]), 0)]               = 1.0
        adm_vec[ADMISSION_MAP.get(str(row["admission_type"]), 1)] = 1.0
        view_vec[CXR_VIEW_MAP.get(str(row["cxr_view"]), 0)]      = 1.0
        return np.concatenate([[age_norm, gender], race_vec, adm_vec, view_vec])

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        embed    = self.embeddings[idx]
        labels, mask = self._encode_labels_and_mask(row)
        metadata = self._encode_metadata(row)
        return {
            "embedding": torch.tensor(embed,    dtype=torch.float32),
            "labels"   : torch.tensor(labels,   dtype=torch.float32),
            "mask"     : torch.tensor(mask,      dtype=torch.float32),
            "metadata" : torch.tensor(metadata,  dtype=torch.float32),
            "hadm_id"  : int(row["hadm_id"]),
        }


print("✅ [FIX 20] Infraestructura de caché de embeddings definida.")
print("   compute_embeddings_cache() + CachedEmbeddingDataset")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 8: MASKED BINARY CROSS-ENTROPY (CONVENIO CHEXPERT)
# ══════════════════════════════════════════════════════════════════════════════

class MaskedBCELoss(nn.Module):
    def __init__(self, pos_weights_dict=None, labels=LABELS):
        super().__init__()
        if pos_weights_dict is not None:
            w = torch.tensor([pos_weights_dict.get(l, 1.0) for l in labels], dtype=torch.float32)
        else:
            w = torch.ones(len(labels), dtype=torch.float32)
        self.register_buffer("pos_weights", w)

    def forward(self, logits, labels, mask):
        bce = F.binary_cross_entropy_with_logits(
            logits, labels, pos_weight=self.pos_weights.to(logits.device), reduction="none"
        )
        masked = bce * mask
        return masked.sum() / mask.sum().clamp(min=1e-8)


criterion = MaskedBCELoss(pos_weights_dict=POS_WEIGHTS).to(DEVICE)
print("✅ MaskedBCELoss lista.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 9: ARQUITECTURA — ResNet-50 + Correlación + Metadatos
# [FIX 10] Colapso 3→1 canal antes del backbone (CheXpert espera grayscale)
# [FIX 11] Detección dinámica de la dimensión de salida del backbone
# [FIX 12] Backbone puro extraído quitando avgpool+fc del modelo XRV completo
# [FIX 16] El backbone se congela y NUNCA se descongela en v3
# [FIX 20] CXRHead: módulo separado que opera sobre embeddings cacheados,
#          reutilizable tanto en modo "imagen completa" (CXRResNet50) como
#          en modo "embedding cacheado" (entrenamiento rápido del Nested CV)
# ══════════════════════════════════════════════════════════════════════════════

class LabelCorrelationModule(nn.Module):
    """Capa lineal NxN aprendible que modela co-ocurrencias entre etiquetas."""
    def __init__(self, n_labels=N_LABELS):
        super().__init__()
        self.correlation = nn.Linear(n_labels, n_labels, bias=False)
        nn.init.eye_(self.correlation.weight)
        self.correlation.weight.data *= 0.1
        self.norm = nn.LayerNorm(n_labels)

    def forward(self, logits):
        return self.norm(logits + self.correlation(logits))


class MetadataBranch(nn.Module):
    """MLP para procesar metadatos clínicos."""
    def __init__(self, meta_dim=META_DIM, meta_embed=META_EMBED):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(meta_dim, 128), nn.BatchNorm1d(128), nn.ReLU(inplace=True), nn.Dropout(0.2),
            nn.Linear(128, meta_embed), nn.BatchNorm1d(meta_embed), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)


def _infer_backbone_out_dim(backbone, img_size=IMG_SIZE):
    """
    [FIX 11] Mide la dimensión real de salida del backbone con un forward dummy.
    """
    was_training = backbone.training
    backbone.eval()
    with torch.no_grad():
        dummy = torch.zeros(2, 1, img_size, img_size)
        out = backbone(dummy)
        if out.ndim == 4:
            out = F.adaptive_avg_pool2d(out, 1).flatten(1)
        elif out.ndim == 3:
            out = out.mean(dim=-1)
        dim = out.shape[1]
    if was_training:
        backbone.train()
    return dim, (out.ndim != 2)


class CXRHead(nn.Module):
    """
    [FIX 20] Cabeza clasificadora separada del backbone. Opera sobre un
    embedding [B, backbone_dim] ya calculado (en vivo o cacheado).
    Esto permite reutilizar la misma cabeza tanto si el embedding viene de
    un forward en tiempo real como de un caché pre-computado.
    """
    def __init__(self, backbone_dim=2048, n_labels=N_LABELS, dropout_rate=0.5,
                 use_label_correlation=True, use_meta_branch=True):
        super().__init__()
        self.use_meta_branch       = use_meta_branch
        self.use_label_correlation = use_label_correlation

        self.img_proj = nn.Sequential(
            nn.Linear(backbone_dim, IMG_PROJ), nn.BatchNorm1d(IMG_PROJ), nn.ReLU(inplace=True)
        )

        if use_meta_branch:
            self.meta_branch = MetadataBranch()
            fusion_dim = IMG_PROJ + META_EMBED
        else:
            self.meta_branch = None
            fusion_dim = IMG_PROJ

        self.head = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(fusion_dim, 256), nn.BatchNorm1d(256), nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(256, n_labels),
        )
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None: nn.init.zeros_(m.bias)

        self.label_corr = LabelCorrelationModule(n_labels) if use_label_correlation else None

    def forward(self, embedding, metadata=None):
        proj = self.img_proj(embedding)
        if self.use_meta_branch and metadata is not None:
            fused = torch.cat([proj, self.meta_branch(metadata)], dim=1)
        else:
            fused = proj
        logits = self.head(fused)
        if self.label_corr is not None:
            logits = self.label_corr(logits)
        return {"logits": logits, "probs": torch.sigmoid(logits)}


class CXRResNet50(nn.Module):
    """
    Modelo completo: backbone (congelado, [FIX 16]) + CXRHead.
    Mantiene compatibilidad con el flujo de imagen completa (necesario para
    el caché inicial de embeddings, FIX 20) y delega la clasificación en
    CXRHead, que es la única parte con gradiente activo.
    """
    def __init__(self, n_labels=N_LABELS, dropout_rate=0.5,
                 use_label_correlation=True, use_meta_branch=True,
                 pretrained_source="chexpert"):
        super().__init__()

        if pretrained_source == "chexpert" and XRV_AVAILABLE:
            try:
                xrv_model = xrv.models.ResNet(weights="resnet50-res512-all")
                full      = xrv_model.model
                # [FIX 12] Extraer backbone puro quitando avgpool+fc
                self.backbone = nn.Sequential(*list(full.children())[:-2])
                print("   ✓ Backbone: ResNet-50 pesos CheXpert (torchxrayvision) [FIX 12]")
            except Exception as e:
                print(f"   ⚠ XRV falló ({e}) → ResNet-50 ImageNet")
                backbone      = tv_models.resnet50(weights="IMAGENET1K_V1")
                self.backbone = nn.Sequential(*list(backbone.children())[:-2])
        else:
            backbone      = tv_models.resnet50(weights="IMAGENET1K_V1")
            self.backbone = nn.Sequential(*list(backbone.children())[:-2])
            print("   ✓ Backbone: ResNet-50 ImageNet")

        backbone_dim, self._needs_gap = _infer_backbone_out_dim(self.backbone, IMG_SIZE)
        print(f"   ✓ Backbone out dim: {backbone_dim}  |  needs_GAP: {self._needs_gap}  [FIX 11]")

        # [FIX 16] Backbone SIEMPRE congelado en v3 — feature extraction puro
        for p in self.backbone.parameters():
            p.requires_grad = False
        self.backbone.eval()
        print("   🔒 [FIX 16] Backbone congelado de forma PERMANENTE (no se descongela en v3).")

        self.cxr_head = CXRHead(
            backbone_dim=backbone_dim, n_labels=n_labels, dropout_rate=dropout_rate,
            use_label_correlation=use_label_correlation, use_meta_branch=use_meta_branch,
        )
        self.label_corr  = self.cxr_head.label_corr   # alias por compatibilidad con código previo
        self.img_proj    = self.cxr_head.img_proj
        self.head         = self.cxr_head.head
        self.meta_branch  = self.cxr_head.meta_branch

    def freeze_backbone(self):
        """[FIX 16] No-op: el backbone ya está congelado permanentemente."""
        pass

    def unfreeze_layers(self, strategy="none"):
        """[FIX 16] No-op: en v3 el backbone nunca se descongela."""
        pass

    @torch.no_grad()
    def extract_embedding(self, image):
        """Calcula el embedding del backbone (sin gradiente, siempre congelado)."""
        if image.shape[1] == 3:
            image = image.mean(dim=1, keepdim=True)  # [FIX 10]
        out = self.backbone(image)
        if out.ndim == 4:
            out = F.adaptive_avg_pool2d(out, 1).flatten(1)
        elif out.ndim == 3:
            out = out.mean(dim=-1)
        return out

    def forward(self, image, metadata=None):
        embedding = self.extract_embedding(image)
        return self.cxr_head(embedding, metadata)


print("✅ Arquitectura CXRResNet50 + CXRHead definida.")
print("   [FIX 16] Backbone congelado permanentemente")
print("   [FIX 20] CXRHead separable para entrenamiento sobre embeddings cacheados")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 10: UTILIDADES — MÉTRICAS, UMBRALES, OPTIMIZADOR
# [FIX 3] ReduceLROnPlateau sin verbose=False
# [FIX 4] num_epochs_override (irrelevante con backbone fijo pero se conserva
#         por si en el futuro se reactiva el fine-tuning)
# [FIX 5] macro_AUC seguro ante listas vacías
# ══════════════════════════════════════════════════════════════════════════════

def compute_multilabel_metrics(all_probs, all_labels, all_masks,
                                thresholds=None, labels=LABELS):
    if thresholds is None:
        thresholds = np.full(len(labels), 0.5)
    metrics, auc_list, ap_list, f1_list = {}, [], [], []
    for i, lbl in enumerate(labels):
        vm     = all_masks[:, i] == 1
        y_true = all_labels[vm, i]
        y_prob = all_probs[vm, i]
        y_pred = (y_prob >= thresholds[i]).astype(float)
        n_pos  = int(y_true.sum())
        n_neg  = int((1 - y_true).sum())
        if n_pos < 2 or n_neg < 2:
            auc, ap = float("nan"), float("nan")
        else:
            auc = roc_auc_score(y_true, y_prob)
            ap  = average_precision_score(y_true, y_prob)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        metrics[lbl] = {"AUC": auc, "AP": ap, "F1": f1, "n_pos": n_pos, "n_neg": n_neg}
        if not np.isnan(auc):
            auc_list.append(auc); ap_list.append(ap)
        f1_list.append(f1)
    # [FIX 5]
    metrics["macro_AUC"] = float(np.nanmean(auc_list)) if auc_list else float("nan")
    metrics["macro_AP"]  = float(np.nanmean(ap_list))  if ap_list  else float("nan")
    metrics["macro_F1"]  = float(np.nanmean(f1_list))  if f1_list  else float("nan")
    return metrics


def find_optimal_thresholds(all_probs, all_labels, all_masks,
                             labels=LABELS, n_thresholds=50):
    thresholds = np.full(len(labels), 0.5)
    for i, lbl in enumerate(labels):
        vm     = all_masks[:, i] == 1
        y_true = all_labels[vm, i]
        y_prob = all_probs[vm, i]
        if y_true.sum() < 2:
            continue
        best_f1, best_thr = -1.0, 0.5
        for thr in np.linspace(0.1, 0.9, n_thresholds):
            f1 = f1_score(y_true, (y_prob >= thr).astype(float), zero_division=0)
            if f1 > best_f1:
                best_f1, best_thr = f1, thr
        thresholds[i] = best_thr
    return thresholds


def build_optimizer_and_scheduler(head_module, config, train_loader_len,
                                   num_epochs_override=None):
    """
    [FIX 16] Construye el optimizador SOLO sobre los parámetros de la cabeza
    (head_module = CXRHead), ya que el backbone está congelado permanentemente.
    [FIX 4] num_epochs_override preservado por compatibilidad futura.
    """
    num_epochs = num_epochs_override if num_epochs_override is not None else config["num_epochs"]

    optimizer = torch.optim.AdamW(
        head_module.parameters(), lr=config["lr_head"], weight_decay=config["weight_decay"]
    )

    sched = config["scheduler"]
    if sched == "cosine":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=num_epochs, eta_min=1e-7)
    elif sched == "onecycle":
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=config["lr_head"] * 10,
            steps_per_epoch=train_loader_len, epochs=num_epochs, pct_start=0.1)
    elif sched == "plateau":
        # [FIX 3] verbose=False eliminado
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=3)
    else:
        scheduler = None
    return optimizer, scheduler


print("✅ Utilidades definidas.")
print("   [FIX 16] Optimizador construido SOLO sobre CXRHead (backbone fuera del grafo)")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 11: ENTRENAMIENTO Y EVALUACIÓN SOBRE EMBEDDINGS CACHEADOS
# [FIX 20] train_one_epoch_cached / evaluate_cached operan sobre CachedEmbeddingDataset,
#          NO sobre imágenes — el backbone ya no se ejecuta en este punto.
# [FIX 17] num_epochs=3, paciencia=1.
# [FIX 8] Progreso con print+flush.
# ══════════════════════════════════════════════════════════════════════════════

def train_one_epoch_cached(head_module, loader, optimizer, scheduler, criterion, config):
    head_module.train()
    total_loss, n = 0.0, 0
    for batch in loader:
        embed    = batch["embedding"].to(DEVICE)
        labels   = batch["labels"].to(DEVICE)
        masks    = batch["mask"].to(DEVICE)
        metadata = batch["metadata"].to(DEVICE) if config.get("use_meta_branch") else None

        optimizer.zero_grad()
        out  = head_module(embed, metadata)
        loss = criterion(out["logits"], labels, masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(head_module.parameters(), max_norm=1.0)
        optimizer.step()

        if config.get("scheduler") == "onecycle" and scheduler is not None:
            scheduler.step()

        total_loss += loss.item(); n += 1
    return total_loss / max(n, 1)


@torch.no_grad()
def evaluate_cached(head_module, loader, criterion, config):
    head_module.eval()
    total_loss, n = 0.0, 0
    probs_l, labels_l, masks_l = [], [], []
    for batch in loader:
        embed    = batch["embedding"].to(DEVICE)
        labels   = batch["labels"].to(DEVICE)
        masks    = batch["mask"].to(DEVICE)
        metadata = batch["metadata"].to(DEVICE) if config.get("use_meta_branch") else None
        out  = head_module(embed, metadata)
        loss = criterion(out["logits"], labels, masks)
        total_loss += loss.item(); n += 1
        probs_l.append(out["probs"].cpu().numpy())
        labels_l.append(labels.cpu().numpy())
        masks_l.append(masks.cpu().numpy())
    return (total_loss / max(n, 1),
            np.concatenate(probs_l), np.concatenate(labels_l), np.concatenate(masks_l))


def train_model_cached(embeddings_train, df_train_fold,
                       embeddings_val, df_val_fold, config, verbose=True):
    """
    [FIX 20] Entrena SOLO la cabeza (CXRHead) sobre embeddings pre-calculados.
    No hay backbone en el bucle de entrenamiento → coste ínfimo en CPU.
    [FIX 17] num_epochs reducido + paciencia=1 (antes 5).

    Returns:
        (best_state_dict, best_val_metrics, best_thresholds, history)
    """
    ds_train = CachedEmbeddingDataset(df_train_fold, embeddings_train,
                                      uncertainty_policy=config["uncertainty_policy"])
    ds_val   = CachedEmbeddingDataset(df_val_fold,   embeddings_val,
                                      uncertainty_policy=config["uncertainty_policy"])

    loader_train = DataLoader(ds_train, batch_size=config["batch_size"],
                              shuffle=True,  num_workers=0, drop_last=True)
    loader_val   = DataLoader(ds_val,   batch_size=config["batch_size"] * 4,
                              shuffle=False, num_workers=0)

    backbone_dim = embeddings_train.shape[1]
    head = CXRHead(
        backbone_dim=backbone_dim,
        dropout_rate=config["dropout_rate"],
        use_label_correlation=config["use_label_correlation"],
        use_meta_branch=config["use_meta_branch"],
    ).to(DEVICE)

    optimizer, scheduler = build_optimizer_and_scheduler(head, config, len(loader_train))

    best_val_loss    = float("inf")
    best_state       = None
    best_metrics     = None
    best_thresholds  = np.full(N_LABELS, 0.5)
    patience_counter = 0
    PATIENCE         = 1   # [FIX 17] antes 5

    history = {"train_loss": [], "val_loss": [], "val_auc_macro": []}

    for epoch in range(config["num_epochs"]):
        train_loss = train_one_epoch_cached(head, loader_train, optimizer, scheduler, criterion, config)
        val_loss, all_probs, all_labels, all_masks = evaluate_cached(head, loader_val, criterion, config)
        val_metrics = compute_multilabel_metrics(all_probs, all_labels, all_masks)

        if scheduler is not None and config["scheduler"] != "onecycle":
            if config["scheduler"] == "plateau":
                scheduler.step(val_loss)
            else:
                scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_auc_macro"].append(val_metrics["macro_AUC"])

        if verbose:
            print(f"       Ep{epoch+1}/{config['num_epochs']}  "
                  f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
                  f"AUC={val_metrics['macro_AUC']:.4f}", flush=True)

        if val_loss < best_val_loss - 1e-4:
            best_val_loss    = val_loss
            best_state       = copy.deepcopy(head.state_dict())
            patience_counter = 0
            if config.get("threshold_search"):
                best_thresholds = find_optimal_thresholds(all_probs, all_labels, all_masks)
            best_metrics = compute_multilabel_metrics(
                all_probs, all_labels, all_masks, thresholds=best_thresholds)
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                if verbose:
                    print(f"       ⏹ Early stopping en época {epoch+1}.", flush=True)
                break

    return best_state, best_metrics, best_thresholds, history


print("✅ [FIX 20] Funciones de entrenamiento sobre embeddings cacheados definidas.")
print("   [FIX 17] PATIENCE=1, num_epochs=3 (configurable en el grid)")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 11b: PRE-CÁLCULO DE EMBEDDINGS DEL BACKBONE (UNA SOLA VEZ)
# [FIX 20] Esta es la celda que materializa la optimización principal de v3.
#          Se instancia el backbone CheXpert UNA VEZ, se calculan los
#          embeddings de TODO el train/val/test, y a partir de aquí el
#          Nested CV opera exclusivamente sobre estos vectores [N, 2048].
#
#          Tiempo esperado: este paso SÍ pasa por el backbone completo
#          (es inevitable, hay que extraer features de cada imagen al menos
#          una vez), pero se hace solo 1 vez en todo el notebook en vez de
#          en cada uno de los 54+ entrenamientos del Nested CV.
# ══════════════════════════════════════════════════════════════════════════════

print("═" * 70)
print("  [FIX 20] PRE-CÁLCULO DE EMBEDDINGS — backbone ejecutado UNA SOLA VEZ")
print("═" * 70)

# Instanciar un modelo temporal solo para extraer el backbone congelado
_temp_model = CXRResNet50(pretrained_source="chexpert")
_backbone   = _temp_model.backbone
_backbone.eval()

t0 = time.time()

print("\n  Calculando embeddings de TRAIN...", flush=True)
embeddings_train_full = compute_embeddings_cache(
    df_train, cxr_train_npy, _backbone, batch_size=16, desc="Embeddings train"
)
print(f"  ✓ embeddings_train_full: {embeddings_train_full.shape}  "
      f"({(time.time()-t0)/60:.1f} min)", flush=True)

t1 = time.time()
print("\n  Calculando embeddings de TEST (evaluación final)...", flush=True)
embeddings_test_full = compute_embeddings_cache(
    df_test, cxr_test_npy, _backbone, batch_size=16, desc="Embeddings test"
)
print(f"  ✓ embeddings_test_full: {embeddings_test_full.shape}  "
      f"({(time.time()-t1)/60:.1f} min)", flush=True)

del _temp_model, _backbone
gc.collect()

print(f"\n✅ [FIX 20] Embeddings pre-calculados en {(time.time()-t0)/60:.1f} min totales.")
print(f"   A partir de aquí, el Nested CV NO vuelve a tocar el backbone.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 12: NESTED CROSS-VALIDATION SOBRE EMBEDDINGS CACHEADOS
# [FIX 5] Guardia NaN  |  [FIX 6] gc.collect()+empty_cache
# [FIX 8] Progreso explícito con print+flush+ETA
# [FIX 14] Subsampling al 15% del train en el loop INTERNO (solo tuning)
# [FIX 18] K_OUTER=2, N_RANDOM_CONFIGS=3 (ver celda 5)
# [FIX 20] Entrenamiento sobre embeddings → cada "entrenamiento" tarda
#          segundos en vez de minutos, ya que no hay backbone en el bucle.
# ══════════════════════════════════════════════════════════════════════════════

K_OUTER = 2   # [FIX 18] antes 3
K_INNER = 3

df_cv     = df_train.copy().reset_index(drop=True)
n_samples = len(df_cv)

# [FIX 14] Fracción de train_outer usada en el loop INTERNO (selección de hiperparámetros).
# El reentrenamiento final (tras elegir la mejor config) SIEMPRE usa el 100%.
INNER_SUBSAMPLE_FRAC = 0.15

print("═" * 70)
print("  NESTED CROSS-VALIDATION — ResNet-50 CXR (v3, sobre embeddings)")
print("═" * 70)
print(f"  Muestras en CV         : {n_samples:,}")
print(f"  K externo               : {K_OUTER}  |  K interno : {K_INNER}")
print(f"  Configs RS              : {len(SAMPLED_CONFIGS)}")
print(f"  [FIX 14] Subsample inner: {INNER_SUBSAMPLE_FRAC:.0%} del train_outer")
print(f"  Total trains internos   : {K_OUTER * len(SAMPLED_CONFIGS) * K_INNER}  "
      f"(sobre embeddings, ~segundos cada uno)")
print(f"  Total reentrenam. final : {K_OUTER}  (100% de datos, también sobre embeddings)")
print("═" * 70, flush=True)

outer_results = []
outer_kf      = KFold(n_splits=K_OUTER, shuffle=True, random_state=SEED)

run_total   = K_OUTER * len(SAMPLED_CONFIGS) * K_INNER
run_current = 0
t_global    = time.time()

for outer_fold_idx, (outer_train_idx, outer_test_idx) in enumerate(
    outer_kf.split(np.arange(n_samples))
):
    print(f"\n{'═'*70}", flush=True)
    print(f"  FOLD EXTERNO {outer_fold_idx+1}/{K_OUTER}  "
          f"(train={len(outer_train_idx):,} | test={len(outer_test_idx):,})", flush=True)
    print(f"{'═'*70}", flush=True)

    df_outer_train = df_cv.iloc[outer_train_idx].reset_index(drop=True)
    df_outer_test  = df_cv.iloc[outer_test_idx].reset_index(drop=True)
    embed_outer_train = embeddings_train_full[outer_train_idx]
    embed_outer_test  = embeddings_train_full[outer_test_idx]

    best_inner_auc = -1.0
    best_config    = SAMPLED_CONFIGS[0]
    inner_kf       = KFold(n_splits=K_INNER, shuffle=True, random_state=SEED)

    for config_idx, config in enumerate(SAMPLED_CONFIGS):
        t_cfg = time.time()
        print(f"\n  ┌─ Config {config_idx+1}/{len(SAMPLED_CONFIGS)} "
              f"[fold ext {outer_fold_idx+1}/{K_OUTER}] {'─'*25}", flush=True)
        print(f"  │  lr_head={config['lr_head']}  dropout={config['dropout_rate']}  "
              f"corr={config['use_label_correlation']}  policy={config['uncertainty_policy']}",
              flush=True)

        auc_scores = []

        for inner_fold_idx, (inner_train_idx, inner_val_idx) in enumerate(
            inner_kf.split(np.arange(len(df_outer_train)))
        ):
            run_current += 1
            t_inner = time.time()

            elapsed   = time.time() - t_global
            avg_per   = elapsed / run_current if run_current > 1 else 0
            remaining = avg_per * (run_total - run_current)

            print(f"  │  [{run_current:3d}/{run_total}] fold interno {inner_fold_idx+1}/{K_INNER}  "
                  f"ETA: {int(remaining//60)}m {int(remaining%60):02d}s ...",
                  end=" ", flush=True)

            df_inner_train_full = df_outer_train.iloc[inner_train_idx].reset_index(drop=True)
            df_inner_val         = df_outer_train.iloc[inner_val_idx].reset_index(drop=True)
            embed_inner_train_full = embed_outer_train[inner_train_idx]
            embed_inner_val         = embed_outer_train[inner_val_idx]

            # [FIX 14] Subsampling al 15% SOLO en el loop interno (tuning)
            n_sub = max(int(len(df_inner_train_full) * INNER_SUBSAMPLE_FRAC), 50)
            sub_idx = np.random.RandomState(SEED + run_current).choice(
                len(df_inner_train_full), size=n_sub, replace=False
            )
            df_inner_train    = df_inner_train_full.iloc[sub_idx].reset_index(drop=True)
            embed_inner_train = embed_inner_train_full[sub_idx]

            _, val_metrics, _, _ = train_model_cached(
                embed_inner_train, df_inner_train,
                embed_inner_val,   df_inner_val,
                config, verbose=False
            )

            # [FIX 5] Guardia NaN
            auc = (val_metrics.get("macro_AUC", float("nan"))
                   if val_metrics is not None else float("nan"))
            auc_scores.append(auc)

            t_inner_s = time.time() - t_inner
            print(f"✓ AUC={auc:.4f}  ({t_inner_s:.1f}s)  [n_train_sub={n_sub}]", flush=True)

            # [FIX 6]
            gc.collect()

        mean_auc  = float(np.nanmean(auc_scores))
        t_cfg_min = (time.time() - t_cfg) / 60
        print(f"  └─ Config {config_idx+1} completada  mean_AUC={mean_auc:.4f}  "
              f"({t_cfg_min:.1f} min)", flush=True)

        if mean_auc > best_inner_auc:
            best_inner_auc = mean_auc
            best_config    = config
            print(f"     ⭐ Nueva mejor config (AUC={best_inner_auc:.4f})", flush=True)

    print(f"\n  ✅ Mejor config fold externo {outer_fold_idx+1}  AUC_inner={best_inner_auc:.4f}",
          flush=True)
    for k, v in best_config.items():
        print(f"     {k:30s} = {v}", flush=True)

    # ── Reentrenamiento final: 100% de train_outer (sin subsampling) ───────────
    print(f"\n  🔁 Reentrenando con mejor config sobre el 100% de train_outer "
          f"({len(df_outer_train):,} muestras)...", flush=True)
    t_final = time.time()

    best_state, _, best_thresholds, history = train_model_cached(
        embed_outer_train, df_outer_train,
        embed_outer_test,  df_outer_test,
        best_config, verbose=True
    )
    print(f"  ✓ Reentrenamiento completado en {(time.time()-t_final):.1f}s", flush=True)

    # ── Evaluación en test externo ─────────────────────────────────────────────
    backbone_dim = embed_outer_train.shape[1]
    head_final = CXRHead(
        backbone_dim=backbone_dim,
        dropout_rate=best_config["dropout_rate"],
        use_label_correlation=best_config["use_label_correlation"],
        use_meta_branch=best_config["use_meta_branch"],
    ).to(DEVICE)
    head_final.load_state_dict(best_state)

    ds_test_o     = CachedEmbeddingDataset(df_outer_test, embed_outer_test,
                                           uncertainty_policy=best_config["uncertainty_policy"])
    loader_test_o = DataLoader(ds_test_o, batch_size=32, shuffle=False, num_workers=0)

    _, test_probs, test_labels_arr, test_masks = evaluate_cached(
        head_final, loader_test_o, criterion, best_config
    )
    test_metrics = compute_multilabel_metrics(
        test_probs, test_labels_arr, test_masks, thresholds=best_thresholds
    )

    print(f"\n  📊 Test externo fold {outer_fold_idx+1}:", flush=True)
    print(f"     macro_AUC={test_metrics['macro_AUC']:.4f}  macro_F1={test_metrics['macro_F1']:.4f}",
          flush=True)
    for lbl in LABELS:
        m = test_metrics[lbl]
        auc_s = f"{m['AUC']:.4f}" if not np.isnan(m["AUC"]) else "  N/A"
        print(f"     {lbl:22s} | AUC={auc_s:>6} | F1={m['F1']:.4f} | "
              f"N+={m['n_pos']:>4} | N-={m['n_neg']:>4}", flush=True)

    ckpt_path = OUTPUT_DIR / f"model_outer_fold{outer_fold_idx+1}.pt"
    torch.save({
        "head_state_dict": best_state,
        "best_config"     : best_config,
        "thresholds"      : best_thresholds.tolist(),
        "test_metrics"    : test_metrics,
        "outer_fold"      : outer_fold_idx + 1,
        "backbone_dim"    : backbone_dim,
    }, ckpt_path)
    print(f"  💾 Checkpoint: {ckpt_path}", flush=True)

    outer_results.append({
        "outer_fold"    : outer_fold_idx + 1,
        "best_config"   : best_config,
        "best_inner_auc": best_inner_auc,
        "test_metrics"  : test_metrics,
        "thresholds"    : best_thresholds.tolist(),
        "history"       : history,
        "checkpoint"    : str(ckpt_path),
    })

    gc.collect()

# ── Resumen final ──────────────────────────────────────────────────────────────
auc_macros = [r["test_metrics"]["macro_AUC"] for r in outer_results]
f1_macros  = [r["test_metrics"]["macro_F1"]  for r in outer_results]

t_total_min = (time.time() - t_global) / 60
print("\n" + "═"*70, flush=True)
print("  NESTED CV COMPLETADO (v3 — sobre embeddings cacheados)", flush=True)
print("═"*70, flush=True)
print(f"  Tiempo total       : {t_total_min:.1f} min", flush=True)
print(f"  AUC macro por fold : {[f'{a:.4f}' for a in auc_macros]}", flush=True)
print(f"  Media AUC macro    : {np.mean(auc_macros):.4f} ± {np.std(auc_macros):.4f}", flush=True)
print(f"  Media F1  macro    : {np.mean(f1_macros):.4f}  ± {np.std(f1_macros):.4f}", flush=True)
print("═"*70, flush=True)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 13: RESUMEN NESTED CV + GUARDADO JSON
# [FIX 7] auc_macros recalculada por robustez de scope
# ══════════════════════════════════════════════════════════════════════════════

auc_macros = [r["test_metrics"]["macro_AUC"] for r in outer_results]
f1_macros  = [r["test_metrics"]["macro_F1"]  for r in outer_results]

print("  AUC por etiqueta (media ± std sobre folds externos):")
for lbl in LABELS:
    aucs = [r["test_metrics"][lbl]["AUC"] for r in outer_results]
    print(f"    {lbl:22s}: {np.nanmean(aucs):.4f} ± {np.nanstd(aucs):.4f}")

summary = {
    "version"       : "v3 (CPU-optimized, embeddings cacheados)",
    "mean_macro_AUC": float(np.mean(auc_macros)),
    "std_macro_AUC" : float(np.std(auc_macros)),
    "mean_macro_F1" : float(np.mean(f1_macros)),
    "std_macro_F1"  : float(np.std(f1_macros)),
    "fold_results"  : [{
        "outer_fold" : r["outer_fold"],
        "macro_AUC"  : r["test_metrics"]["macro_AUC"],
        "macro_F1"   : r["test_metrics"]["macro_F1"],
        "best_config": r["best_config"],
        "thresholds" : r["thresholds"],
        "checkpoint" : r["checkpoint"],
        "per_label"  : {lbl: r["test_metrics"][lbl] for lbl in LABELS},
    } for r in outer_results],
}
p = OUTPUT_DIR / "nested_cv_summary_v3.json"
with open(p, "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"\n  💾 Resumen guardado: {p}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 14: VISUALIZACIÓN — CURVAS Y AUC POR ETIQUETA
# ══════════════════════════════════════════════════════════════════════════════

auc_macros = [r["test_metrics"]["macro_AUC"] for r in outer_results]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("ResNet-50 CXR v3 (CPU-optimized) — Nested CV Results", fontsize=13, fontweight="bold")

ax = axes[0]
for r in outer_results:
    h = r["history"]
    ax.plot(h["train_loss"], linestyle="--", alpha=0.5, label=f"Fold {r['outer_fold']} train")
    ax.plot(h["val_loss"],                  alpha=0.9, label=f"Fold {r['outer_fold']} val")
ax.set_xlabel("Época"); ax.set_ylabel("Loss"); ax.set_title("Pérdida por Fold")
ax.legend(fontsize=7); ax.grid(alpha=0.3)

ax = axes[1]
for r in outer_results:
    ax.plot(r["history"]["val_auc_macro"], alpha=0.9, label=f"Fold {r['outer_fold']}")
ax.axhline(np.mean(auc_macros), color="red", linestyle=":", label="Media")
ax.set_xlabel("Época"); ax.set_ylabel("AUC macro"); ax.set_title("AUC macro en Val")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[2]
means = [np.nanmean([r["test_metrics"][l]["AUC"] for r in outer_results]) for l in LABELS]
stds  = [np.nanstd( [r["test_metrics"][l]["AUC"] for r in outer_results]) for l in LABELS]
bars  = ax.bar(range(len(LABELS)), means, yerr=stds, color="steelblue", alpha=0.7, capsize=4)
ax.set_xticks(range(len(LABELS)))
ax.set_xticklabels([l[:10] for l in LABELS], rotation=35, ha="right", fontsize=8)
ax.set_ylabel("AUC-ROC"); ax.set_title("AUC por Etiqueta")
ax.set_ylim([0, 1.05])
ax.axhline(np.mean(auc_macros), color="red", linestyle="--", alpha=0.7, label="Macro media")
ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y")
for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.01, f"{val:.3f}",
            ha="center", fontsize=7)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "nested_cv_results_v3.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Figura guardada.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 15: EVALUACIÓN FINAL EN TEST SET OFICIAL
# [FIX 7] recalculo de auc_macros  |  [FIX 20] usa embeddings_test_full pre-calculados
# ⚠ EJECUTAR SOLO UNA VEZ.
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "═"*70)
print("  EVALUACIÓN FINAL EN TEST SET OFICIAL")
print("  ⚠ Esta evaluación se ejecuta UNA SOLA VEZ.")
print("═"*70)

auc_macros = [r["test_metrics"]["macro_AUC"] for r in outer_results]

best_outer_fold = outer_results[int(np.argmax(auc_macros))]
print(f"\n  Fold seleccionado: {best_outer_fold['outer_fold']}  "
      f"(AUC_outer={auc_macros[int(np.argmax(auc_macros))]:.4f})")

ckpt         = torch.load(best_outer_fold["checkpoint"], map_location=DEVICE)
best_cfg     = ckpt["best_config"]
backbone_dim = ckpt["backbone_dim"]

head_test = CXRHead(
    backbone_dim=backbone_dim,
    dropout_rate=best_cfg["dropout_rate"],
    use_label_correlation=best_cfg["use_label_correlation"],
    use_meta_branch=best_cfg["use_meta_branch"],
).to(DEVICE)
head_test.load_state_dict(ckpt["head_state_dict"])

best_thresholds_test = np.array(ckpt["thresholds"])

# [FIX 20] usar embeddings de test ya pre-calculados en celda 11b
ds_test_off = CachedEmbeddingDataset(df_test, embeddings_test_full,
                                     uncertainty_policy=best_cfg["uncertainty_policy"])
loader_test = DataLoader(ds_test_off, batch_size=32, shuffle=False, num_workers=0)

print(f"  Test samples: {len(ds_test_off):,}")

_, test_probs_f, test_labels_f, test_masks_f = evaluate_cached(
    head_test, loader_test, criterion, best_cfg
)
test_metrics_final = compute_multilabel_metrics(
    test_probs_f, test_labels_f, test_masks_f, thresholds=best_thresholds_test
)

print("\n  📊 MÉTRICAS FINALES:")
print(f"  {'Etiqueta':22s} | {'AUC':>6} | {'AP':>6} | {'F1':>6} | {'N+':>5} | {'N-':>5}")
print("  " + "─"*60)
for lbl in LABELS:
    m = test_metrics_final[lbl]
    auc_s = f"{m['AUC']:.4f}" if not np.isnan(m["AUC"]) else "  N/A "
    ap_s  = f"{m['AP']:.4f}"  if not np.isnan(m["AP"])  else "  N/A "
    print(f"  {lbl:22s} | {auc_s:>6} | {ap_s:>6} | {m['F1']:>6.4f} | {m['n_pos']:>5} | {m['n_neg']:>5}")
print("  " + "─"*60)
print(f"  {'MACRO':22s} | {test_metrics_final['macro_AUC']:>6.4f} | "
      f"{test_metrics_final['macro_AP']:>6.4f} | {test_metrics_final['macro_F1']:>6.4f}")

with open(OUTPUT_DIR / "final_test_results_v3.json", "w") as f:
    json.dump({
        "macro_AUC" : test_metrics_final["macro_AUC"],
        "macro_AP"  : test_metrics_final["macro_AP"],
        "macro_F1"  : test_metrics_final["macro_F1"],
        "per_label" : {lbl: test_metrics_final[lbl] for lbl in LABELS},
        "best_config": best_cfg,
    }, f, indent=2, default=str)
print(f"\n  💾 Resultados finales guardados.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 16: ANÁLISIS DE EQUIDAD POR SUBGRUPO
# ══════════════════════════════════════════════════════════════════════════════

df_test_r = df_test.reset_index(drop=True)

for col, vals, label in [
    ("gender",         [0, 1],                       "GÉNERO"),
    ("cxr_view",       ["AP", "PA"],                 "VISTA CXR"),
    ("race",           list(RACE_MAP.keys()),        "RAZA"),
    ("admission_type", list(ADMISSION_MAP.keys()),   "TIPO ADMISIÓN"),
]:
    print(f"\n── AUC macro por {label} ──────────────────────────────────────────")
    for v in vals:
        idx = df_test_r[df_test_r[col] == v].index.values
        if len(idx) < 10:
            print(f"  {str(v):25s}: n={len(idx)} (insuficiente)")
            continue
        m = compute_multilabel_metrics(
            test_probs_f[idx], test_labels_f[idx], test_masks_f[idx],
            thresholds=best_thresholds_test
        )
        print(f"  {str(v):25s} (n={len(idx):4d}): AUC={m['macro_AUC']:.4f}  F1={m['macro_F1']:.4f}")

print("\n✅ Análisis de equidad completado.")


---
## ✅ Resumen completo de fixes acumulados (v1 → v3)

| Fix | Celda | Descripción |
|-----|-------|-------------|
| **[FIX 1]**  | 4, 7  | Imágenes leídas desde `.npy`, no JPGs en disco |
| **[FIX 2]**  | 4, 7  | Join CSV↔npy por `hadm_id` → `_npy_idx` |
| **[FIX 2b]** | 4, 15 | Test CSV=464 vs npy=4640 — resuelto por el join |
| **[FIX 3]**  | 10    | `ReduceLROnPlateau` sin `verbose=False` |
| **[FIX 4]**  | 10, 11| `num_epochs_override` preservado |
| **[FIX 5]**  | 10, 12| Guardia NaN en `macro_AUC` |
| **[FIX 6]**  | 12    | `gc.collect()` + `cuda.empty_cache()` |
| **[FIX 7]**  | 13,14,15| Recalculo de `auc_macros` por scope |
| **[FIX 8]**  | 11,12 | Progreso con `print(..., flush=True)` + ETA |
| **[FIX 9]**  | 6, 7  | Sin `A.Normalize` — npy ya normalizado |
| **[FIX 10]** | 9, 7b | Colapso 3→1 canal antes del backbone |
| **[FIX 11]** | 9     | Detección dinámica de dimensión del backbone |
| **[FIX 12]** | 9     | Backbone puro sin `avgpool+fc` |
| **[FIX 13]** | 3     | `IMG_SIZE` 320→**160** |
| **[FIX 14]** | 12    | Subsampling al **15%** en loop interno |
| **[FIX 15]** | 5     | `batch_size` fijado a **8** |
| **[FIX 16]** | 5,9,10| Backbone **siempre congelado** (feature extraction puro) |
| **[FIX 17]** | 5,11  | `num_epochs=3`, paciencia=**1** |
| **[FIX 18]** | 5,12  | `N_RANDOM_CONFIGS=3`, `K_OUTER=2` |
| **[FIX 19]** | 2     | `torch.set_num_threads(4)` — núcleos físicos reales |
| **[FIX 20]** | 7b,9,11,11b,12,15 | **Caché de embeddings**: backbone ejecutado 1 sola vez, Nested CV entrena solo la cabeza |

## ⚠️ Limitaciones a documentar en la memoria del TFM
1. El backbone permanece **congelado** (no fine-tuning) — es "feature extraction" puro, una simplificación forzada por el hardware disponible (CPU sin GPU, 4 núcleos).
2. El caché de embeddings ([FIX 20]) implica que la augmentación geométrica/de intensidad **no se aplica** durante el entrenamiento de la cabeza (se calculó el embedding sobre la imagen sin augmentación aleatoria).
3. El subsampling al 15% en el loop interno reduce la robustez de la selección de hiperparámetros, aunque el reentrenamiento final siempre usa el 100% de los datos.
4. `K_OUTER=2` es el mínimo razonable para nested CV; idealmente se reportaría con K≥5 si se dispone de más cómputo (ej. ejecutando en Google Colab con GPU).

## 📌 Próximos pasos
1. Si el TFM lo permite, ejecutar esta misma lógica en Google Colab (GPU gratuita) para reportar resultados con el grid completo de la v2.
2. Aplicar la misma estrategia de caché de embeddings al modelo ResNet1D de ECG.
3. XGBoost para labs tabulares (no requiere GPU, coste mínimo).
4. Late fusion stacking de los 3 modelos base.
